# **Para quais países doar?**

**Contexto.** Você é a pessoa mais rica do mundo e tem um orçamento filantrópico finito.

O dataset traz pib per capita mas não o número de habitantes

não sei a data do dataset, mas pelos dados acho que algo próximo de 2010 no terremoto do Haiti

##**Imports e Configs**

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.stats import spearmanr

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')

# ---- Paleta: uma única família de azul, do claro ao escuro, porque a variável que
# ---- estamos codificando (necessidade) é uma magnitude ordenada, não categorias soltas.
SUPERFICIE = '#fcfcfb'
TINTA      = '#0b0b0b'
TINTA_2    = '#52514e'
TINTA_3    = '#898781'
GRADE      = '#e1e0d9'

AZUL_CLARO = '#86b6ef'
AZUL       = '#2a78d6'
AZUL_ESC   = '#104281'
DESTAQUE   = '#eb6834'

CORES_CLUSTER = {'Desenvolvido': AZUL_CLARO, 'Em desenvolvimento': AZUL, 'Crítico': AZUL_ESC}
ESCALA_SEQ = ['#cde2fb', '#9ec5f4', '#6da7ec', '#3987e5', '#2a78d6', '#184f95', '#0d366b']
ESCALA_DIV = ['#0d366b', '#2a78d6', '#9ec5f4', '#f0efec', '#f2a3a2', '#e34948', '#a12b2a']

pio.templates['doacoes'] = go.layout.Template(layout=dict(
    paper_bgcolor=SUPERFICIE, plot_bgcolor=SUPERFICIE,
    font=dict(family='system-ui, -apple-system, Segoe UI, sans-serif', size=13, color=TINTA_2),
    title=dict(font=dict(size=17, color=TINTA)),
    xaxis=dict(gridcolor=GRADE, zerolinecolor=GRADE, linecolor='#c3c2b7', tickfont=dict(color=TINTA_3)),
    yaxis=dict(gridcolor=GRADE, zerolinecolor=GRADE, linecolor='#c3c2b7', tickfont=dict(color=TINTA_3)),
    legend=dict(font=dict(color=TINTA_2)),
    colorway=[AZUL, DESTAQUE, '#1baf7a'],
    margin=dict(t=70, r=30, b=55, l=60),
))
pio.templates.default = 'doacoes'
try:
    import google.colab
    pio.renderers.default = 'colab'
except ImportError:
    pio.renderers.default = 'notebook'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Carregando o arquivo CSV do Google Drive
file_path = '/content/drive/MyDrive/01_TRABALHO/ESTAGIO/projetos_cd/Clustering/dataset_paises.csv'
df = pd.read_csv(file_path)
print(f"Arquivo '{file_path}' carregado com sucesso!")

# Carregando o dicionario do Google Drive
file_path2 = '/content/drive/MyDrive/01_TRABALHO/ESTAGIO/projetos_cd/Clustering/dicionario_dataset_paises.csv'
dicionario = pd.read_csv(file_path2)
print(f"Arquivo '{file_path2}' carregado com sucesso!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Arquivo '/content/drive/MyDrive/01_TRABALHO/ESTAGIO/projetos_cd/Clustering/dataset_paises.csv' carregado com sucesso!
Arquivo '/content/drive/MyDrive/01_TRABALHO/ESTAGIO/projetos_cd/Clustering/dicionario_dataset_paises.csv' carregado com sucesso!


## **Primeira visão**

In [ ]:
display(dicionario)

,Nome da Coluna,Descrição
0,pais,Nome do país
1,mortalidade_infantil,Mortes de crianças menores de 5 anos por 1.000...
2,exportacoes,"Exportações de bens e serviços, fornecidas com..."
3,saude,Gastos totais com saúde como porcentagem (%) d...
4,importacoes,"Importações de bens e serviços, fornecidas com..."
5,renda,Renda líquida por pessoa
6,inflacao,Medida da taxa anual de crescimento do PIB total
7,expectativa_vida,Média de anos que um recém-nascido viveria se ...
8,fertilidade_total,Número de filhos que nasceriam de cada mulher ...
9,pib_per_capita,"O PIB per capita, calculado como o PIB total d..."


In [ ]:
print(df.head())
print(df.tail())

                  pais  mortalidade_infantil  exportacoes  saude  importacoes  renda  inflacao  expectativa_vida  fertilidade_total  pib_per_capita
0          Afghanistan                 90.20        10.00   7.58        44.90   1610      9.44             56.20               5.82             553
1              Albania                 16.60        28.00   6.55        48.60   9930      4.49             76.30               1.65            4090
2              Algeria                 27.30        38.40   4.17        31.40  12900     16.10             76.50               2.89            4460
3               Angola                119.00        62.30   2.85        42.90   5900     22.40             60.10               6.16            3530
4  Antigua and Barbuda                 10.30        45.50   6.03        58.90  19100      1.44             76.80               2.13           12200
          pais  mortalidade_infantil  exportacoes  saude  importacoes  renda  inflacao  expectativa_vida  fertil

In [ ]:
print('--- Tipos e completude ---')
print(df.dtypes)
print(f'\nValores nulos no dataset inteiro: {df.isna().sum().sum()}')
print(f'Países duplicados: {df["pais"].duplicated().sum()}')

--- Tipos e completude ---
pais                     object
mortalidade_infantil    float64
exportacoes             float64
saude                   float64
importacoes             float64
renda                     int64
inflacao                float64
expectativa_vida        float64
fertilidade_total       float64
pib_per_capita            int64
dtype: object

Valores nulos no dataset inteiro: 0
Países duplicados: 0


* países puxando a média da mortalidade infantil para cima assim como exportações, renda e pib_per_capita

In [ ]:
numericas = df.select_dtypes(include=np.number).columns.tolist()
print(f'\nVariáveis numéricas: {len(numericas)}')
df[numericas].describe()


Variáveis numéricas: 9


,mortalidade_infantil,exportacoes,saude,importacoes,renda,inflacao,expectativa_vida,fertilidade_total,pib_per_capita
count,167.00,167.00,167.00,167.00,167.00,167.00,167.00,167.00,167.00
mean,38.27,41.11,6.82,46.89,"17,144.69",7.78,70.56,2.95,"12,964.16"
std,40.33,27.41,2.75,24.21,"19,278.07",10.57,8.89,1.51,"18,328.70"
min,2.60,0.11,1.81,0.07,609.00,-4.21,32.10,1.15,231.00
25%,8.25,23.80,4.92,30.20,"3,355.00",1.81,65.30,1.79,"1,330.00"
50%,19.30,35.00,6.32,43.30,"9,960.00",5.39,73.10,2.41,"4,660.00"
75%,62.10,51.35,8.60,58.75,"22,800.00",10.75,76.80,3.88,"14,050.00"
max,208.00,200.00,17.90,174.00,"125,000.00",104.00,82.80,7.49,"105,000.00"


pode-se ter problemas de escala, a Renda varia de 609 a 125.000, enquanto a expectativa de vida varia de 32 a 83

In [ ]:
assimetria = df[numericas].skew().sort_values(ascending=False)
print('--- Assimetria (skew) ---')
print(assimetria.round(2))
print('\nAcima de 1 em módulo = cauda longa, candidata a transformação logarítmica.\n')

print('--- Outliers pelo critério IQR (1,5 x amplitude interquartil) ---')
for coluna in numericas:
    q1, q3 = df[coluna].quantile([0.25, 0.75])
    iqr = q3 - q1
    fora = df.loc[(df[coluna] < q1 - 1.5*iqr) | (df[coluna] > q3 + 1.5*iqr), 'pais'].tolist()
    print(f'{coluna:22s} {len(fora):3d}  {", ".join(fora[:6])}{" ..." if len(fora) > 6 else ""}')

--- Assimetria (skew) ---
inflacao                5.15
exportacoes             2.45
renda                   2.23
pib_per_capita          2.22
importacoes             1.91
mortalidade_infantil    1.45
fertilidade_total       0.97
saude                   0.71
expectativa_vida       -0.97
dtype: float64

Acima de 1 em módulo = cauda longa, candidata a transformação logarítmica.

--- Outliers pelo critério IQR (1,5 x amplitude interquartil) ---
mortalidade_infantil     4  Central African Republic, Chad, Haiti, Sierra Leone
exportacoes              5  Ireland, Luxembourg, Malta, Seychelles, Singapore
saude                    2  Micronesia, Fed. Sts., United States
importacoes              4  Luxembourg, Malta, Seychelles, Singapore
renda                    8  Brunei, Kuwait, Luxembourg, Norway, Qatar, Singapore ...
inflacao                 5  Equatorial Guinea, Mongolia, Nigeria, Timor-Leste, Venezuela
expectativa_vida         3  Central African Republic, Haiti, Lesotho
fertilidade_total   

O dicionário descreve errado a inflação, já que os valores mostram a nigéria com 104, venezuela com 45,9 e por aí vai, assim como valores negativos na irlanda e japão, como é definido como 'taxa de crescimento anual do PIB total' é de se desconfiar pois, nenhuma economia cresce 104% ao ano, mas já tiveram a inflação por aí.

saúde aparece com ogasto de saúde como porcentual do pib

## **Definindo os critérios de necessidade**

In [ ]:
titulos = ['Mortalidade infantil', 'Exportações (% PIB)', 'Gasto com saúde (% PIB)',
           'Importações (% PIB)', 'Renda por pessoa', 'Inflação',
           'Expectativa de vida', 'Fertilidade total', 'PIB per capita']
fig = make_subplots(rows=3, cols=3, subplot_titles=titulos, vertical_spacing=0.12, horizontal_spacing=0.08)

for i, coluna in enumerate(numericas):
    fig.add_trace(go.Histogram(x=df[coluna], marker_color=AZUL, marker_line_width=0,
                               name=coluna, showlegend=False,
                               hovertemplate='faixa %{x}<br>%{y} países<extra></extra>'),
                  row=i//3 + 1, col=i%3 + 1)

fig.update_layout(height=760, bargap=0.06, title_text='Distribuição das nove variáveis originais',
                  title_x=0.02, margin=dict(t=90))
fig.update_annotations(font=dict(size=12, color=TINTA_2))
fig.show()

* Cauda longa em PIB per capita, inflação, exportações e importações
* MOrtalidade infantil tendo 86 países na primeira faixa
* Fertilidade Bimodal ( 1,8 filho por mulher, depressão perto de 4, e uma segunda elevação por volta de 5) grupo mais visível
* Cauda a esquerda na expectativa de vida

In [ ]:
rotulos_curtos = ['mort. infantil', 'exportações', 'saúde', 'importações', 'renda',
                  'inflação', 'expect. vida', 'fertilidade', 'PIB per capita']
corr = df[numericas].corr()
corr.index = rotulos_curtos
corr.columns = rotulos_curtos

fig = px.imshow(corr, text_auto='.2f', aspect='auto', zmin=-1, zmax=1,
                color_continuous_scale=ESCALA_DIV,
                labels=dict(color='Correlação'),
                title='Matriz de correlação de Pearson')
fig.update_traces(textfont=dict(size=11), xgap=2, ygap=2,
                  hovertemplate='%{y} vs %{x}<br>r = %{z:.2f}<extra></extra>')
fig.update_layout(height=640, coloraxis_colorbar=dict(thickness=14, len=0.7),
                  margin=dict(l=130, t=70, r=30, b=120))
fig.update_xaxes(tickangle=-40, tickfont=dict(size=11, color=TINTA_2))
fig.update_yaxes(tickfont=dict(size=11, color=TINTA_2))
fig.show()

* Mortalidade infantil e expectativa de vida correlacionam a -0.89
* Mortalidade e fertilidade: 0.85
* Fertilidade e expectativa -0.76
* renda e PIB per capita a 0.90
* renda e mortalidade a -0.52
* renda e expectativa a 0.61


* exportações e importações a 0.74

podemos ver que mortalidade infantil, expectativa de vida, fetilidade, renda e PIB, se mostrando como variáveis fortes para se trabalhar.

In [ ]:
fig = px.scatter(df, x='renda', y='mortalidade_infantil', log_x=True,
                 color='expectativa_vida', color_continuous_scale=ESCALA_SEQ[::-1],
                 hover_name='pais', size='fertilidade_total', size_max=22,
                 labels=dict(renda='Renda líquida por pessoa (escala log)',
                             mortalidade_infantil='Mortes por 1.000 nascidos vivos (< 5 anos)',
                             expectativa_vida='Expectativa<br>de vida'),
                 title='Renda e mortalidade infantil')
fig.update_traces(marker=dict(line=dict(width=1.2, color=SUPERFICIE)),
                  hovertemplate='<b>%{hovertext}</b><br>renda %{x:,.0f}<br>mortalidade %{y:.1f}<extra></extra>')

posicao_rotulo = {'Haiti': (40, -22), 'Sierra Leone': (46, -22), 'Central African Republic': (-20, -34),
                  'Niger': (-34, 26), 'Norway': (-16, -40), 'Qatar': (-34, -30)}
for nome, (dx, dy) in posicao_rotulo.items():
    linha = df[df.pais == nome].iloc[0]
    fig.add_annotation(x=np.log10(linha.renda), y=linha.mortalidade_infantil, text=nome,
                       showarrow=True, arrowhead=0, arrowwidth=1, arrowcolor=TINTA_3,
                       ax=dx, ay=dy, font=dict(size=11, color=TINTA))

fig.update_layout(height=560, coloraxis_colorbar=dict(thickness=14, len=0.7),
                  xaxis=dict(dtick=1, minor=dict(showgrid=False)))
fig.show()

In [ ]:
fig = px.scatter(df, x='fertilidade_total', y='expectativa_vida',
                 color='mortalidade_infantil', color_continuous_scale=ESCALA_SEQ,
                 hover_name='pais',
                 labels=dict(fertilidade_total='Filhos por mulher',
                             expectativa_vida='Expectativa de vida (anos)',
                             mortalidade_infantil='Mortalidade<br>infantil'),
                 title='Transição demográfica')
fig.update_traces(marker=dict(size=12, line=dict(width=1, color=TINTA_3)),
                  hovertemplate='<b>%{hovertext}</b><br>fertilidade %{x:.2f}<br>expectativa %{y:.1f} anos'
                                '<br>mortalidade %{marker.color:.0f} por 1.000<extra></extra>')
fig.add_vrect(x0=3.0, x1=4.5, fillcolor=DESTAQUE, opacity=0.09, line_width=0,
              annotation_text='zona de transição: 22 países em 1,5 filho de amplitude', annotation_position='top',
              annotation_font=dict(color=DESTAQUE, size=12))
fig.update_layout(height=560, coloraxis_colorbar=dict(thickness=14, len=0.7))
fig.show()

onde muitas crianças nascem, muitas também morrem, e a expectativa de vida também é menor

In [ ]:
corrige_nome = {
    'Congo, Dem. Rep.': 'Democratic Republic of the Congo',
    'Congo, Rep.': 'Republic of the Congo',
    'Macedonia, FYR': 'North Macedonia',
    'Micronesia, Fed. Sts.': 'Micronesia',
    'Lao': 'Laos',
    "Cote d'Ivoire": 'Ivory Coast',
    'St. Vincent and the Grenadines': 'Saint Vincent and the Grenadines',
    'Cape Verde': 'Cabo Verde',
    'Slovak Republic': 'Slovakia',
    'Kyrgyz Republic': 'Kyrgyzstan',
    'Czech Republic': 'Czechia',
    'Antigua and Barbuda': 'Antigua and Barb.',
}
df['pais_mapa'] = df['pais'].replace(corrige_nome)

fig = px.choropleth(df, locations='pais_mapa', locationmode='country names',
                    color='mortalidade_infantil', hover_name='pais',
                    color_continuous_scale=ESCALA_SEQ,
                    labels=dict(mortalidade_infantil='Mortes por<br>1.000 nasc.'),
                    title='Mortalidade infantil no mundo')
fig.update_traces(marker_line_color=SUPERFICIE, marker_line_width=0.6,
                  hovertemplate='<b>%{hovertext}</b><br>%{z:.1f} mortes por 1.000<extra></extra>')
fig.update_geos(showframe=False, showcoastlines=False, projection_type='natural earth',
                bgcolor=SUPERFICIE, landcolor='#f0efec')
fig.update_layout(height=520, coloraxis_colorbar=dict(thickness=14, len=0.7))
fig.show()

Sinais de carência está concentrada na áfria subsaariana, com alguns pontos no sul da ásia

In [ ]:
df['faixa_renda'] = pd.qcut(df['renda'], 5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])
agrupado = df.groupby('faixa_renda', observed=True)[numericas].mean().round(2)
print('--- Média dos indicadores por quintil de renda ---')
display(agrupado)

indicadores = ['mortalidade_infantil', 'expectativa_vida', 'fertilidade_total', 'saude']
rotulos = ['Mortalidade infantil<br>(por 1.000)', 'Expectativa de vida<br>(anos)', 'Fertilidade<br>(filhos/mulher)', 'Gasto com saúde<br>(% do PIB)']
fig = make_subplots(rows=1, cols=4, subplot_titles=rotulos, horizontal_spacing=0.06)

for i, coluna in enumerate(indicadores):
    valores = agrupado[coluna]
    fig.add_trace(go.Bar(x=list(agrupado.index), y=valores, showlegend=False,
                         marker_color=[AZUL_ESC, AZUL, AZUL, AZUL_CLARO, AZUL_CLARO],
                         marker_line_width=2, marker_line_color=SUPERFICIE,
                         text=[f'{v:.1f}' for v in valores], textposition='outside',
                         textfont=dict(size=11, color=TINTA_2),
                         hovertemplate='%{x}<br>%{y:.2f}<extra></extra>'),
                  row=1, col=i+1)

fig.update_layout(height=470, title_text='O gradiente de renda',
                  title_x=0.02, margin=dict(t=110, b=90))
fig.update_xaxes(tickangle=0, tickfont=dict(size=12, color=TINTA_2))
fig.update_annotations(font=dict(size=12, color=TINTA_2))
fig.add_annotation(xref='paper', yref='paper', x=0.5, y=-0.17, showarrow=False,
                   text='quintil de renda (Q1 = mais pobre, Q5 = mais rico)',
                   font=dict(size=12, color=TINTA_3))
fig.show()

--- Média dos indicadores por quintil de renda ---


,mortalidade_infantil,exportacoes,saude,importacoes,renda,inflacao,expectativa_vida,fertilidade_total,pib_per_capita
faixa_renda,,,,,,,,,
Q1,91.93,22.05,6.89,45.64,"1,565.09",7.81,59.32,4.99,743.41
Q2,53.28,36.02,5.59,46.81,"4,783.64",12.16,67.05,3.52,"2,155.70"
Q3,23.61,38.13,6.52,43.77,"10,611.82",7.91,72.05,2.36,"5,200.00"
Q4,12.76,51.48,6.90,51.10,"20,306.06",6.28,75.02,1.85,"13,021.21"
Q5,9.02,57.93,8.13,47.15,"47,994.12",4.85,79.41,2.00,"43,155.88"


O gasto com saúde em países com menor renda é maior do que até daqueles com renda média

In [ ]:
linhas = []
for faixa in ['Q1', 'Q2', 'Q3', 'Q4', 'Q5']:
    grupo = df[df.faixa_renda == faixa].sort_values('renda')
    linhas.append({
        'quintil': faixa,
        'n_paises': len(grupo),
        'renda_min': grupo.renda.iloc[0],
        'renda_max': grupo.renda.iloc[-1],
        'mais_pobre': grupo.pais.iloc[0],
        'mais_rico': grupo.pais.iloc[-1],
        'mortalidade_media': round(grupo.mortalidade_infantil.mean(), 2),
    })

print('--- Composição de cada quintil de renda ---')
display(pd.DataFrame(linhas).set_index('quintil'))

print('\n--- Os 3 primeiros e os 3 últimos países de cada quintil ---')
for faixa in ['Q1', 'Q2', 'Q3', 'Q4', 'Q5']:
    ordenado = df[df.faixa_renda == faixa].sort_values('renda')['pais'].tolist()
    print(f'{faixa}: {", ".join(ordenado[:3])}  ...  {", ".join(ordenado[-3:])}')

--- Composição de cada quintil de renda ---


,n_paises,renda_min,renda_max,mais_pobre,mais_rico,mortalidade_media
quintil,,,,,,
Q1,34,609,2520,"Congo, Dem. Rep.",Cambodia,91.93
Q2,33,2660,7300,Cameroon,El Salvador,53.28
Q3,33,7350,14500,Fiji,Brazil,23.61
Q4,33,15300,28700,Bulgaria,Slovenia,12.76
Q5,34,29600,125000,Israel,Qatar,9.02



--- Os 3 primeiros e os 3 últimos países de cada quintil ---
Q1: Congo, Dem. Rep., Liberia, Burundi  ...  Bangladesh, Kenya, Cambodia
Q2: Cameroon, Cote d'Ivoire, Kyrgyz Republic  ...  Georgia, Paraguay, El Salvador
Q3: Fiji, Mongolia, Ukraine  ...  Montenegro, Suriname, Brazil
Q4: Bulgaria, Barbados, Gabon  ...  Czech Republic, Greece, Slovenia
Q5: Israel, Libya, South Korea  ...  Brunei, Luxembourg, Qatar


In [ ]:
PAIS = 'Brazil'
i = df.index[df.pais == PAIS][0]

print('=' * 78)
print(f'PERFIL DE {PAIS.upper()} NO CONJUNTO DE 167 PAÍSES')
print('=' * 78)
print(f"Grupo do clustering : {df.loc[i, 'grupo']}")
print(f"Quintil de renda    : {df.loc[i, 'faixa_renda']}")
print(f"Severidade          : {df.loc[i, 'severidade']:+.3f}   "
      f"(posição {int(df.severidade.rank(ascending=False)[i])}º de 167 — 1º é o mais carente)")

print('\n--- Cada indicador: valor, posição e distância da média mundial ---')
perfil = pd.DataFrame({
    'valor_brasil': df.loc[i, numericas],
    'media_mundial': df[numericas].mean(),
    'mediana_mundial': df[numericas].median(),
    'percentil': (df[numericas].rank(pct=True).loc[i] * 100).round(1),
    'posicao': df[numericas].rank(ascending=False).loc[i].astype(int).astype(str) + 'º',
})
display(perfil.round(2))

print('\n--- Brasil x média do próprio grupo x média dos 10 recomendados ---')
top10 = df.nlargest(10, 'severidade').index
comparacao = pd.DataFrame({
    'Brasil': df.loc[i, CONJUNTO_B],
    'Média do grupo do Brasil': df[df.grupo == df.loc[i, 'grupo']][CONJUNTO_B].mean(),
    'Média mundial': df[CONJUNTO_B].mean(),
    'Média do Top-10 carente': df.loc[top10, CONJUNTO_B].mean(),
})
display(comparacao.round(2))

print('\n--- Os 6 países mais parecidos com o Brasil (distância no espaço padronizado) ---')
distancia = np.linalg.norm(X_B - X_B[i], axis=1)
vizinhos = df.assign(distancia=distancia).nsmallest(7, 'distancia').iloc[1:]
display(vizinhos[['pais', 'grupo', 'distancia', 'mortalidade_infantil',
                  'expectativa_vida', 'renda', 'severidade']].round(3))

PERFIL DE BRAZIL NO CONJUNTO DE 167 PAÍSES
Grupo do clustering : Desenvolvido
Quintil de renda    : Q3
Severidade          : -0.475   (posição 100º de 167 — 1º é o mais carente)

--- Cada indicador: valor, posição e distância da média mundial ---


,valor_brasil,media_mundial,mediana_mundial,percentil,posicao
mortalidade_infantil,19.80,38.27,19.30,51.50,82º
exportacoes,10.70,41.11,35.00,4.20,161º
saude,9.01,6.82,6.32,78.70,36º
importacoes,11.80,46.89,43.30,1.20,166º
renda,14500,"17,144.69","9,960.00",59.90,68º
inflacao,8.41,7.78,5.39,66.50,57º
expectativa_vida,74.20,70.56,73.10,57.50,72º
fertilidade_total,1.80,2.95,2.41,25.70,125º
pib_per_capita,11200,"12,964.16","4,660.00",68.90,53º



--- Brasil x média do próprio grupo x média dos 10 recomendados ---


,Brasil,Média do grupo do Brasil,Média mundial,Média do Top-10 carente
mortalidade_infantil,19.80,8.32,38.27,135.16
expectativa_vida,74.20,77.85,70.56,53.47
fertilidade_total,1.80,1.82,2.95,5.78
renda,14500,"32,612.96","17,144.69","1,190.30"
pib_per_capita,11200,"26,912.25","12,964.16",499.10



--- Os 6 países mais parecidos com o Brasil (distância no espaço padronizado) ---


,pais,grupo,distancia,mortalidade_infantil,expectativa_vida,renda,severidade
100,Mauritius,Desenvolvido,0.32,15.00,73.40,15900,-0.48
124,Romania,Desenvolvido,0.37,11.50,73.70,17800,-0.54
71,Iran,Desenvolvido,0.39,19.30,74.50,17400,-0.47
13,Barbados,Desenvolvido,0.40,14.20,76.70,15300,-0.63
131,Seychelles,Desenvolvido,0.41,14.40,73.40,20400,-0.52
160,Uruguay,Desenvolvido,0.41,10.60,76.40,17100,-0.62


## **Pré-processamento**

In [ ]:
transformado = df.copy()
for coluna in ['renda', 'pib_per_capita', 'exportacoes', 'importacoes']:
    transformado[coluna] = np.log1p(transformado[coluna])
transformado['inflacao'] = np.log1p(transformado['inflacao'] - transformado['inflacao'].min() + 1)

antes = df[numericas].skew().round(2)
depois = transformado[numericas].skew().round(2)
comparacao = pd.DataFrame({'skew antes': antes, 'skew depois': depois})
comparacao['melhorou'] = comparacao['skew depois'].abs() < comparacao['skew antes'].abs()
print('--- Efeito da transformação logarítmica ---')
display(comparacao)
print('A inflação foi deslocada antes do log porque tem valores negativos (deflação).')

--- Efeito da transformação logarítmica ---


,skew antes,skew depois,melhorou
mortalidade_infantil,1.45,1.45,False
exportacoes,2.45,-1.09,True
saude,0.71,0.71,False
importacoes,1.91,-1.82,True
renda,2.23,-0.24,True
inflacao,5.15,0.45,True
expectativa_vida,-0.97,-0.97,False
fertilidade_total,0.97,0.97,False
pib_per_capita,2.22,0.01,True


A inflação foi deslocada antes do log porque tem valores negativos (deflação).


In [ ]:
CONJUNTO_A = ['mortalidade_infantil', 'exportacoes', 'saude', 'importacoes', 'renda',
              'inflacao', 'expectativa_vida', 'fertilidade_total', 'pib_per_capita']
CONJUNTO_B = ['mortalidade_infantil', 'expectativa_vida', 'fertilidade_total', 'renda', 'pib_per_capita']

X_A = StandardScaler().fit_transform(transformado[CONJUNTO_A])
X_B = StandardScaler().fit_transform(transformado[CONJUNTO_B])

print(f'Conjunto A (todas as 9 variáveis): {X_A.shape}')
print(f'Conjunto B (só as de necessidade): {X_B.shape}')
print(f'\nApós padronização, média = {X_B.mean():.2e} e desvio = {X_B.std():.2f} em ambos.')

Conjunto A (todas as 9 variáveis): (167, 9)
Conjunto B (só as de necessidade): (167, 5)

Após padronização, média = 1.05e-16 e desvio = 1.00 em ambos.


In [ ]:
pca_diag = PCA().fit(X_B)
individual = pca_diag.explained_variance_ratio_ * 100
acumulada = individual.cumsum()
eixo = [f'PC{i+1}' for i in range(len(individual))]

fig = go.Figure()
fig.add_trace(go.Bar(x=eixo, y=individual, name='Variância do componente',
                     marker_color=AZUL, marker_line_width=2, marker_line_color=SUPERFICIE,
                     text=[f'{v:.1f}%' for v in individual], textposition='outside',
                     textfont=dict(size=11, color=TINTA_2),
                     hovertemplate='%{x}<br>%{y:.1f}% da variância<extra></extra>'))
fig.add_trace(go.Scatter(x=eixo, y=acumulada, name='Variância acumulada', mode='lines+markers',
                         line=dict(color=DESTAQUE, width=2), marker=dict(size=9, line=dict(width=2, color=SUPERFICIE)),
                         hovertemplate='%{x}<br>%{y:.1f}% acumulado<extra></extra>'))
fig.update_layout(height=440, yaxis_title='% da variância total', yaxis_range=[0, 108],
                  title='PCA do conjunto de necessidade: um único eixo carrega quase tudo',
                  legend=dict(orientation='h', y=-0.16))
fig.show()

O primeiro componente principal sozinho explica 84,8% da variância das cinco variáveis de necessidade, ou seja, não descrevem 5 dimensões, mas sim uma única dimensão, o que podemos ler como grau de desenvolvimento humano

Ou seja, a apartir dele, temos o desenvolvimento humano que define um aordem de necessidade muito bem definida.

## **Clustering**

A partir da separação das variáveis, o conjunto A com todas as 9, e o B só com as variáveis de necessidade, usei 3 algorítmos, K-means, Aglomerativo com ligaçãod e Ward, e GMM(Gaussian Mixture Model)

Também pesquisei por 3 métricas de avaliação para comparar como agem na prática, e entender o seu funcionamento


Uma colinha de suas métricas:
- **Silhueta** (quanto maior, melhor, máximo 1): o quão mais perto cada ponto está do seu próprio grupo do que do grupo vizinho.
- **Davies-Bouldin** (quanto menor, melhor): razão entre dispersão interna e separação entre centros.
- **Calinski-Harabasz** (quanto maior, melhor): razão entre variância explicada entre grupos e dentro dos grupos.


In [ ]:
linhas = []
for nome, X in [('A - 9 variáveis', X_A), ('B - necessidade', X_B)]:
    for k in range(2, 9):
        km = KMeans(n_clusters=k, n_init=25, random_state=42).fit(X)
        linhas.append({'conjunto': nome, 'k': k, 'inercia': km.inertia_,
                       'silhueta': silhouette_score(X, km.labels_)})
curva = pd.DataFrame(linhas)

fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.09,
                    subplot_titles=['Cotovelo — conjunto A (9 variáveis)',
                                    'Cotovelo — conjunto B (necessidade)',
                                    'Silhueta — os dois conjuntos'])

for col, nome in [(1, 'A - 9 variáveis'), (2, 'B - necessidade')]:
    sub = curva[curva.conjunto == nome]
    fig.add_trace(go.Scatter(x=sub.k, y=sub.inercia, mode='lines+markers', showlegend=False,
                             line=dict(color=AZUL, width=2), marker=dict(size=9, line=dict(width=2, color=SUPERFICIE)),
                             hovertemplate='k=%{x}<br>inércia %{y:.0f}<extra></extra>'), row=1, col=col)

for nome, cor in [('A - 9 variáveis', DESTAQUE), ('B - necessidade', AZUL)]:
    sub = curva[curva.conjunto == nome]
    fig.add_trace(go.Scatter(x=sub.k, y=sub.silhueta, mode='lines+markers', name=nome,
                             line=dict(color=cor, width=2), marker=dict(size=9, line=dict(width=2, color=SUPERFICIE)),
                             hovertemplate=nome + '<br>k=%{x}<br>silhueta %{y:.3f}<extra></extra>'), row=1, col=3)

fig.add_vline(x=3, line_dash='dot', line_color=TINTA_3, line_width=1.5, row=1, col=2)
fig.update_xaxes(title_text='número de grupos (k)', dtick=1)
fig.update_layout(height=480, title_text='Escolha de k: o cotovelo do conjunto B é nítido em k=3',
                  title_x=0.02, margin=dict(t=110, b=120), legend=dict(orientation='h', y=-0.34, x=0.02))
fig.update_annotations(font=dict(size=12, color=TINTA_2))
fig.show()

display(curva.pivot(index='k', columns='conjunto', values='silhueta').round(3))

conjunto,A - 9 variáveis,B - necessidade
k,,
2,0.35,0.55
3,0.25,0.43
4,0.25,0.39
5,0.22,0.36
6,0.21,0.36
7,0.22,0.37
8,0.20,0.34


Para o grupo B porque não escolher k=2? A divisão fica muito grosseira, apenas entre países desenvolvidos ou não

In [ ]:
resultados = []
for nome, X in [('A - 9 variáveis', X_A), ('B - necessidade', X_B)]:
    for k in [3, 4]:
        modelos = {
            'K-Means': KMeans(n_clusters=k, n_init=25, random_state=42).fit_predict(X),
            'Aglomerativo (Ward)': AgglomerativeClustering(n_clusters=k, linkage='ward').fit_predict(X),
            'GMM': GaussianMixture(n_components=k, n_init=10, random_state=42).fit_predict(X),
        }
        for algoritmo, rotulos_previstos in modelos.items():
            tamanhos = np.bincount(rotulos_previstos)
            resultados.append({
                'conjunto': nome, 'k': k, 'algoritmo': algoritmo,
                'silhueta': round(silhouette_score(X, rotulos_previstos), 3),
                'davies_bouldin': round(davies_bouldin_score(X, rotulos_previstos), 3),
                'calinski_harabasz': round(calinski_harabasz_score(X, rotulos_previstos), 1),
                'menor_grupo': int(tamanhos.min()),
                'tamanhos': ' / '.join(str(int(x)) for x in sorted(tamanhos, reverse=True)),
            })

placar = pd.DataFrame(resultados).sort_values('silhueta', ascending=False).reset_index(drop=True)
placar['silhueta'] = placar['silhueta'].map('{:.3f}'.format)
placar['davies_bouldin'] = placar['davies_bouldin'].map('{:.3f}'.format)
print('--- Grade completa: 2 conjuntos x 3 algoritmos x 2 valores de k ---')
display(placar)

--- Grade completa: 2 conjuntos x 3 algoritmos x 2 valores de k ---


,conjunto,k,algoritmo,silhueta,davies_bouldin,calinski_harabasz,menor_grupo,tamanhos
0,B - necessidade,3,K-Means,0.426,0.808,264.50,40,71 / 56 / 40
1,B - necessidade,3,Aglomerativo (Ward),0.420,0.857,251.00,41,85 / 41 / 41
2,B - necessidade,4,K-Means,0.389,0.885,240.90,32,59 / 39 / 37 / 32
3,B - necessidade,4,Aglomerativo (Ward),0.377,0.888,223.80,33,52 / 41 / 41 / 33
4,B - necessidade,4,GMM,0.356,0.942,208.70,29,63 / 39 / 36 / 29
5,B - necessidade,3,GMM,0.323,0.938,190.00,30,86 / 51 / 30
6,A - 9 variáveis,4,K-Means,0.254,1.067,64.40,1,73 / 49 / 44 / 1
7,A - 9 variáveis,3,K-Means,0.247,1.376,80.20,42,69 / 56 / 42
8,A - 9 variáveis,3,Aglomerativo (Ward),0.241,1.327,71.00,30,89 / 48 / 30
9,A - 9 variáveis,4,Aglomerativo (Ward),0.207,1.491,58.80,30,53 / 48 / 36 / 30


Incluir as variáveis comerciais produz um grupo degenerado
Entre os algoritmos a diferença é pequena

Modelo final: K-means com k=3

In [ ]:
kmeans_final = KMeans(n_clusters=3, n_init=25, random_state=42)
grupo_bruto = kmeans_final.fit_predict(X_B)

# Os rótulos do K-Means são arbitrários. Reordenamos pela mortalidade infantil média
# para que o grupo 2 seja sempre o mais crítico, independente da semente.
ordem = pd.Series(df['mortalidade_infantil'].values).groupby(grupo_bruto).mean().sort_values().index
mapa_ordem = {rotulo_antigo: posicao for posicao, rotulo_antigo in enumerate(ordem)}
nomes = {0: 'Desenvolvido', 1: 'Em desenvolvimento', 2: 'Crítico'}

df['cluster'] = pd.Series(grupo_bruto).map(mapa_ordem)
df['grupo'] = df['cluster'].map(nomes)

perfil = df.groupby('grupo')[CONJUNTO_B + ['saude', 'inflacao']].mean().round(2)
perfil['n_paises'] = df['grupo'].value_counts()
perfil = perfil.loc[['Desenvolvido', 'Em desenvolvimento', 'Crítico']]
print('--- Perfil médio de cada grupo ---')
display(perfil)

--- Perfil médio de cada grupo ---


,mortalidade_infantil,expectativa_vida,fertilidade_total,renda,pib_per_capita,saude,inflacao,n_paises
grupo,,,,,,,,
Desenvolvido,8.32,77.85,1.82,"32,612.96","26,912.25",7.81,4.67,71
Em desenvolvimento,32.67,69.73,2.74,"7,726.25","3,543.07",5.88,8.71,56
Crítico,99.28,58.76,5.25,"2,874.32","1,395.80",6.37,12.00,40


In [ ]:
z_perfil = pd.DataFrame(X_B, columns=CONJUNTO_B).groupby(df['grupo']).mean()
z_perfil = z_perfil.loc[['Desenvolvido', 'Em desenvolvimento', 'Crítico']]

fig = go.Figure()
for grupo in z_perfil.index:
    fig.add_trace(go.Bar(x=[c.replace('_', ' ') for c in CONJUNTO_B], y=z_perfil.loc[grupo],
                         name=f'{grupo} (n={int(perfil.loc[grupo, "n_paises"])})',
                         marker_color=CORES_CLUSTER[grupo], marker_line_width=2, marker_line_color=SUPERFICIE,
                         text=[f'{v:+.2f}' for v in z_perfil.loc[grupo]], textposition='outside',
                         textfont=dict(size=10, color=TINTA_2),
                         hovertemplate=grupo + '<br>%{x}<br>%{y:+.2f} desvios da média mundial<extra></extra>'))

fig.add_hline(y=0, line_color='#c3c2b7', line_width=1.5)
fig.update_layout(height=500, barmode='group', bargap=0.28, bargroupgap=0.06,
                  yaxis_title='desvios-padrão em relação à média mundial', yaxis_range=[-2.0, 2.4],
                  title='Assinatura de cada grupo: o crítico é o espelho invertido do desenvolvido',
                  legend=dict(orientation='h', y=-0.16))
fig.show()

O grupo **Crítico** está acima da média mundial em mortalidade infantil e fertilidade, e abaixo em expectativa de vida, renda e PIB per capita. O grupo **Em desenvolvimento** fica próximo de zero em todas as variáveis, ou seja, é literalmente a média do mundo.

In [ ]:
coords = PCA(n_components=2).fit_transform(X_B)
# O sinal de um componente principal é arbitrário. Invertemos o PC1 para que
# valores maiores signifiquem mais necessidade, e o eixo se leia da esquerda para a direita.
df['pc1'], df['pc2'] = -coords[:, 0], coords[:, 1]

fig = px.scatter(df, x='pc1', y='pc2', color='grupo', hover_name='pais',
                 color_discrete_map=CORES_CLUSTER,
                 category_orders={'grupo': ['Desenvolvido', 'Em desenvolvimento', 'Crítico']},
                 labels=dict(pc1='PC1 — eixo de necessidade, 84,8% da variância (→ mais carente)',
                             pc2='PC2 (8,1%)', grupo='Grupo'),
                 title='Os três grupos projetados nos dois primeiros componentes principais')
fig.update_traces(marker=dict(size=11, line=dict(width=1.5, color=SUPERFICIE)),
                  hovertemplate='<b>%{hovertext}</b><extra></extra>')

posicao_rotulo = {'Haiti': (-40, -26), 'Sierra Leone': (0, -34), 'Niger': (-38, 26),
                  'Equatorial Guinea': (52, -20), 'Luxembourg': (-46, -26), 'Singapore': (-44, 26)}
for nome, (dx, dy) in posicao_rotulo.items():
    linha = df[df.pais == nome].iloc[0]
    fig.add_annotation(x=linha.pc1, y=linha.pc2, text=nome, showarrow=True, arrowhead=0,
                       arrowwidth=1, arrowcolor=TINTA_3, ax=dx, ay=dy,
                       font=dict(size=11, color=TINTA))

fig.update_layout(height=560, legend=dict(orientation='h', y=-0.18))
fig.show()

Existe uma zona de sobreposição entre o grupo crítico e o em desenvolvimento.
 A guiné equatorial está no grupo crítico por ter mortalidade infantil de 111, mas tem renda de 33.700 dólares por pessoa, maior que a de Portugal. O PC2 existe justamente para capturar o descolamento entre a dimensão econômica e a dimensão de saúde, e a Guiné Equatorial é o caso extremo disso.


In [ ]:
fig = px.choropleth(df, locations='pais_mapa', locationmode='country names',
                    color='grupo', hover_name='pais',
                    color_discrete_map=CORES_CLUSTER,
                    category_orders={'grupo': ['Desenvolvido', 'Em desenvolvimento', 'Crítico']},
                    labels=dict(grupo='Grupo'),
                    title='Os três grupos no mapa')
fig.update_traces(marker_line_color=SUPERFICIE, marker_line_width=0.6,
                  hovertemplate='<b>%{hovertext}</b><extra></extra>')
fig.update_geos(showframe=False, showcoastlines=False, projection_type='natural earth',
                bgcolor=SUPERFICIE, landcolor='#f0efec')
fig.update_layout(height=520, legend=dict(orientation='h', y=-0.02))
fig.show()

print('--- Países do grupo Crítico ---')
print(', '.join(sorted(df.loc[df.grupo == 'Crítico', 'pais'])))

--- Países do grupo Crítico ---
Afghanistan, Angola, Benin, Burkina Faso, Burundi, Cameroon, Central African Republic, Chad, Comoros, Congo, Dem. Rep., Congo, Rep., Cote d'Ivoire, Equatorial Guinea, Eritrea, Gambia, Ghana, Guinea, Guinea-Bissau, Haiti, Kenya, Kiribati, Lesotho, Liberia, Madagascar, Malawi, Mali, Mauritania, Mozambique, Niger, Nigeria, Pakistan, Rwanda, Senegal, Sierra Leone, Sudan, Tanzania, Timor-Leste, Togo, Uganda, Zambia


## **Índice de severidade ordenado dentro do grupo crítico**

Com os clusters eu sei onde procurar, em tese sou muito rica, mas caso não desse para ajudar esses 40 países, quem eu começaria ajudando?

precisava somar cinco variáveis em unidades diferentes, e o z-score converte todas para "quantos desvios da média mundial"
Indicação da IA para uso dos Z-scores das variáveis de necessidade calculados sobre os 167 países com o sinal orientado para que o valor  alto sempre signifique ums situação pior.

Sobre os pesos:

Após testes, descobri que graças a correlação de 0.85 entre fertilidade e mortalidade, e de 0.90 entre renda e PIB per capita, sar peso igual as cinco iria contar a carência duas vezes em 2 pares diferentes. Assim como o gasto com saúde continua fora, pois ele já é uma medida que ajuda em uma porção melhor os países em estado crítico. Então com isso em mente, os pesos escolhidos foram:

| Variável | Peso | Justificativa |
|---|---|---|
| Mortalidade infantil | 0,30 | Medida mais direta de morte evitável, e o desfecho que uma doação pode alterar mais rápido |
| Expectativa de vida | 0,25 | Sintetiza a condição de saúde da população inteira, não só da infância |
| Renda | 0,20 | Dimensão econômica, dividida com o PIB per capita porque os dois medem quase o mesmo |
| PIB per capita | 0,15 | Complementa a renda com a capacidade produtiva do país |
| Fertilidade total | 0,10 | Peso menor por ser largamente redundante com a mortalidade infantil |

abaixo o índice é submetido a um teste de sensibilidade contra a alternativa mais óbvia, que é pesar tudo igualmente

In [ ]:
z = pd.DataFrame(X_B, columns=CONJUNTO_B)

df['z_mortalidade'] =  z['mortalidade_infantil']
df['z_expectativa'] = -z['expectativa_vida']
df['z_fertilidade'] =  z['fertilidade_total']
df['z_renda']       = -z['renda']
df['z_pib']         = -z['pib_per_capita']

df['severidade'] = (0.30 * df.z_mortalidade + 0.25 * df.z_expectativa +
                    0.20 * df.z_renda + 0.15 * df.z_pib + 0.10 * df.z_fertilidade)

df['severidade_pesos_iguais'] = df[['z_mortalidade', 'z_expectativa', 'z_fertilidade',
                                    'z_renda', 'z_pib']].mean(axis=1)

print('--- Severidade média por grupo ---')
display(df.groupby('grupo')['severidade'].agg(['mean', 'min', 'max']).round(3)
          .loc[['Desenvolvido', 'Em desenvolvimento', 'Crítico']])

--- Severidade média por grupo ---


,mean,min,max
grupo,,,
Desenvolvido,-0.83,-1.34,-0.43
Em desenvolvimento,0.07,-0.43,0.79
Crítico,1.37,0.63,2.87


In [ ]:
criticos = df[df.grupo == 'Crítico'].sort_values('severidade', ascending=False).copy()
criticos['posicao'] = range(1, len(criticos) + 1)

colunas_tabela = ['posicao', 'pais', 'severidade', 'mortalidade_infantil', 'expectativa_vida',
                  'fertilidade_total', 'renda', 'pib_per_capita']
print('--- Ranking de severidade dentro do grupo crítico (top 20 de 40) ---')
display(criticos[colunas_tabela].head(20).set_index('posicao').round(2))

--- Ranking de severidade dentro do grupo crítico (top 20 de 40) ---


,pais,severidade,mortalidade_infantil,expectativa_vida,fertilidade_total,renda,pib_per_capita
posicao,,,,,,,
1,Haiti,2.87,208.00,32.10,3.33,1500,662
2,Central African Republic,2.25,149.00,47.50,5.21,888,446
3,Sierra Leone,2.08,160.00,55.00,5.20,1220,399
4,Niger,1.92,123.00,58.80,7.49,814,348
5,"Congo, Dem. Rep.",1.90,116.00,57.50,6.54,609,334
6,Chad,1.89,150.00,56.50,6.59,1930,897
7,Mali,1.74,137.00,59.50,6.55,1870,708
8,Mozambique,1.71,101.00,54.50,5.56,918,419
9,Burundi,1.71,93.60,57.70,6.26,764,231


In [ ]:
top15 = criticos.head(15).iloc[::-1]

fig = go.Figure(go.Bar(
    x=top15.severidade, y=top15.pais, orientation='h',
    marker_color=[AZUL_ESC if v >= 1.65 else AZUL for v in top15.severidade],
    marker_line_width=2, marker_line_color=SUPERFICIE,
    text=[f'{v:.2f}' for v in top15.severidade], textposition='outside',
    textfont=dict(size=11, color=TINTA_2),
    customdata=np.stack([top15.mortalidade_infantil, top15.expectativa_vida, top15.renda], axis=1),
    hovertemplate='<b>%{y}</b><br>severidade %{x:.2f}<br>mortalidade %{customdata[0]:.0f} por 1.000'
                  '<br>expectativa %{customdata[1]:.1f} anos<br>renda %{customdata[2]:,.0f}<extra></extra>'))

fig.add_annotation(x=3.15, y='Malawi', text='azul-escuro = Top-10 recomendado', showarrow=False,
                   font=dict(size=11, color=AZUL_ESC), xanchor='right')
fig.add_annotation(x=3.15, y='Guinea', text='azul-claro = 11º ao 15º', showarrow=False,
                   font=dict(size=11, color=AZUL), xanchor='right')
fig.update_layout(height=580, xaxis_title='índice de severidade (desvios-padrão ponderados)',
                  xaxis_range=[0, 3.25], margin=dict(l=205, t=70, r=30, b=60),
                  yaxis=dict(tickfont=dict(size=12, color=TINTA_2)),
                  title='Os 15 países com maior severidade: o Haiti está em uma categoria própria')
fig.show()

In [ ]:
correlacao_rank, p_valor = spearmanr(criticos.severidade, criticos.severidade_pesos_iguais)
criticos['posicao_iguais'] = criticos['severidade_pesos_iguais'].rank(ascending=False).astype(int)

top10_ponderado = set(criticos.head(10).pais)
top10_iguais = set(criticos.nlargest(10, 'severidade_pesos_iguais').pais)

print(f'Correlação de Spearman entre os dois esquemas de peso: {correlacao_rank:.4f}  (p = {p_valor:.2e})')
print(f'Países no Top-10 dos dois esquemas: {len(top10_ponderado & top10_iguais)} de 10')
print(f'Só no Top-10 ponderado: {top10_ponderado - top10_iguais}')
print(f'Só no Top-10 com pesos iguais: {top10_iguais - top10_ponderado}')

fig = px.scatter(criticos, x='posicao_iguais', y='posicao', hover_name='pais',
                 labels=dict(posicao_iguais='posição com pesos iguais', posicao='posição com pesos ponderados'),
                 title=f'Teste de sensibilidade: o ranking praticamente não depende dos pesos (ρ = {correlacao_rank:.3f})')
fig.update_traces(marker=dict(size=10, color=AZUL, line=dict(width=1.5, color=SUPERFICIE)),
                  hovertemplate='<b>%{hovertext}</b><br>ponderado: %{y}º<br>pesos iguais: %{x}º<extra></extra>')
fig.add_trace(go.Scatter(x=[1, 40], y=[1, 40], mode='lines', showlegend=False,
                         line=dict(color=TINTA_3, width=1.5, dash='dot'), hoverinfo='skip'))
fig.update_layout(height=520)
fig.show()

Correlação de Spearman entre os dois esquemas de peso: 0.9876  (p = 3.52e-32)
Países no Top-10 dos dois esquemas: 9 de 10
Só no Top-10 ponderado: {'Guinea-Bissau'}
Só no Top-10 com pesos iguais: {'Burkina Faso'}


In [ ]:
print('--- Onde o índice discorda do clustering: as fronteiras ---')
print('\nMenores severidades DENTRO do grupo crítico:')
display(criticos.nsmallest(5, 'severidade')[['pais', 'severidade', 'renda', 'mortalidade_infantil', 'expectativa_vida']].round(2))

print('\nMaiores severidades FORA do grupo crítico (grupo em desenvolvimento):')
display(df[df.grupo == 'Em desenvolvimento'].nlargest(5, 'severidade')[['pais', 'severidade', 'renda', 'mortalidade_infantil', 'expectativa_vida']].round(2))

--- Onde o índice discorda do clustering: as fronteiras ---

Menores severidades DENTRO do grupo crítico:


,pais,severidade,renda,mortalidade_infantil,expectativa_vida
49,Equatorial Guinea,0.63,33700,111.00,60.90
149,Timor-Leste,0.67,1850,62.60,71.10
38,"Congo, Rep.",0.76,5190,63.90,60.40
142,Sudan,0.82,3370,76.70,66.30
80,Kenya,0.87,2480,62.20,62.80



Maiores severidades FORA do grupo crítico (grupo em desenvolvimento):


,pais,severidade,renda,mortalidade_infantil,expectativa_vida
84,Lao,0.79,3980,78.90,63.80
136,Solomon Islands,0.66,1780,28.10,61.70
146,Tajikistan,0.60,2110,52.40,69.60
165,Yemen,0.58,4480,56.30,67.50
107,Myanmar,0.57,3720,64.40,66.80


## **Conclusão**

Para quais países doar e por quê

In [ ]:
recomendados = criticos.head(10).copy()

tabela_final = recomendados[['posicao', 'pais', 'severidade', 'mortalidade_infantil',
                             'expectativa_vida', 'fertilidade_total', 'renda', 'pib_per_capita']].copy()
tabela_final.columns = ['#', 'País', 'Severidade', 'Mort. infantil', 'Expectativa',
                        'Fertilidade', 'Renda', 'PIB per capita']

print('=' * 96)
print('RECOMENDAÇÃO FINAL — 10 PAÍSES PRIORITÁRIOS PARA DOAÇÃO')
print('=' * 96)
display(tabela_final.set_index('#').round(2))

media_mundial = df[CONJUNTO_B].mean()
media_top10 = recomendados[CONJUNTO_B].mean()
print('\n--- Top-10 recomendado contra a média mundial ---')
comparativo = pd.DataFrame({'Top-10 recomendado': media_top10, 'Média mundial': media_mundial})
comparativo['razão'] = (comparativo['Top-10 recomendado'] / comparativo['Média mundial']).round(2)
display(comparativo.round(2))

RECOMENDAÇÃO FINAL — 10 PAÍSES PRIORITÁRIOS PARA DOAÇÃO


,País,Severidade,Mort. infantil,Expectativa,Fertilidade,Renda,PIB per capita
#,,,,,,,
1,Haiti,2.87,208.00,32.10,3.33,1500,662
2,Central African Republic,2.25,149.00,47.50,5.21,888,446
3,Sierra Leone,2.08,160.00,55.00,5.20,1220,399
4,Niger,1.92,123.00,58.80,7.49,814,348
5,"Congo, Dem. Rep.",1.90,116.00,57.50,6.54,609,334
6,Chad,1.89,150.00,56.50,6.59,1930,897
7,Mali,1.74,137.00,59.50,6.55,1870,708
8,Mozambique,1.71,101.00,54.50,5.56,918,419
9,Burundi,1.71,93.60,57.70,6.26,764,231



--- Top-10 recomendado contra a média mundial ---


,Top-10 recomendado,Média mundial,razão
mortalidade_infantil,135.16,38.27,3.53
expectativa_vida,53.47,70.56,0.76
fertilidade_total,5.78,2.95,1.96
renda,"1,190.30","17,144.69",0.07
pib_per_capita,499.10,"12,964.16",0.04


In [ ]:
# Todos os z-scores abaixo estão orientados na direção da carência: quanto maior, pior.
colunas_z = ['z_mortalidade', 'z_expectativa', 'z_fertilidade', 'z_renda', 'z_pib']
rotulos_z = ['Mortalidade infantil', 'Expectativa de vida', 'Fertilidade total', 'Renda', 'PIB per capita']

comparacao_z = pd.DataFrame({
    'Desenvolvido': df[df.grupo == 'Desenvolvido'][colunas_z].mean(),
    'Em desenvolvimento': df[df.grupo == 'Em desenvolvimento'][colunas_z].mean(),
    'Crítico (40 países)': df[df.grupo == 'Crítico'][colunas_z].mean(),
    'Top-10 recomendado': recomendados[colunas_z].mean(),
})
comparacao_z.index = rotulos_z
display(comparacao_z.round(2))

fig = go.Figure()
for i, rotulo in enumerate(rotulos_z):
    fig.add_trace(go.Scatter(x=[comparacao_z.iloc[i].min(), comparacao_z.iloc[i].max()], y=[rotulo, rotulo],
                             mode='lines', line=dict(color=GRADE, width=6), showlegend=False, hoverinfo='skip'))

for serie, cor in [('Desenvolvido', AZUL_CLARO), ('Em desenvolvimento', AZUL),
                   ('Crítico (40 países)', AZUL_ESC), ('Top-10 recomendado', DESTAQUE)]:
    fig.add_trace(go.Scatter(x=comparacao_z[serie], y=rotulos_z, mode='markers', name=serie,
                             marker=dict(size=15, color=cor, line=dict(width=2, color=SUPERFICIE)),
                             hovertemplate=serie + '<br>%{y}: %{x:+.2f}<extra></extra>'))

fig.add_vline(x=0, line_color='#c3c2b7', line_width=1.5)
fig.add_annotation(xref='paper', yref='paper', x=0.5, y=-0.16, showarrow=False,
                   text='z-score orientado para a carência — à direita do zero é pior que a média mundial',
                   font=dict(size=12, color=TINTA_3))
fig.update_layout(height=520, margin=dict(l=175, t=70, r=40, b=120),
                  xaxis=dict(range=[-1.5, 2.8], title=None),
                  yaxis=dict(tickfont=dict(size=12, color=TINTA_2), autorange='reversed'),
                  title='Os 10 recomendados são a ponta extrema do grupo crítico',
                  legend=dict(orientation='h', y=-0.24, x=0.0))
fig.show()

,Desenvolvido,Em desenvolvimento,Crítico (40 países),Top-10 recomendado
Mortalidade infantil,-0.74,-0.14,1.52,2.41
Expectativa de vida,-0.82,0.09,1.33,1.93
Fertilidade total,-0.75,-0.14,1.52,1.88
Renda,-0.91,0.25,1.27,1.72
PIB per capita,-0.94,0.35,1.17,1.58
